# Dónde se concentra el abandono en la secundaria argentina

**Entregable 3 de la mentoría (Machine Learning)** · Grupo M01, “El Factor D10S” ·
Diplomatura en Ciencia de Datos, FAMAF (UNC), 2026

**Integrantes:** Pablo Micheli, Jorge Luis Mira e Ignacio Villanueva

**Mentora:** Noelia Ferrero

Cruzamos lo que 540.040 estudiantes contaron en las Pruebas Aprender con
los chicos que sus escuelas registraron como abandono.

## El problema y los datos

Queremos saber **en qué grupos de escuelas secundarias se concentra el abandono**, para que
quien tenga que decidir dónde intervenir sepa por dónde empezar. Usamos dos fuentes públicas
del Ministerio de Educación que miran cosas distintas. Las **Pruebas Aprender** son la
evaluación nacional que rinden los estudiantes del último año: además de la prueba,
responden un cuestionario sobre su hogar, su trayectoria y su escuela. El **Relevamiento
Anual** es el formulario que cada escuela completa todos los años: cuenta cuántos chicos se
inscribieron, cuántos pasaron de año y cuántos se fueron. Aprender no puede ver el abandono,
porque el que ya dejó no va a rendir; el Relevamiento sí, pero no sabe nada de los hogares.
La **Encuesta Permanente de Hogares** (EPH, la encuesta oficial del INDEC sobre empleo e
ingresos de los hogares) tampoco sirve para medir el abandono acá: no se puede unir a estos
grupos, porque solo llega a provincia, y su dato es de 2025.

La pregunta de partida era qué estudiante abandona, pero Aprender no se publica estudiante
por estudiante sino por **grupo de escuelas**: un departamento de una provincia, separando
escuelas estatales y privadas, urbanas y rurales. Son 1.174 grupos con
540.040 estudiantes, y todo lo que sigue habla de esos grupos, nunca de un
chico en particular. Llamamos **perfil** de un grupo a lo que cuentan sus estudiantes en
Aprender sobre su hogar, su trayectoria y su escuela. Como abandono usamos las **salidas sin
pase**: estudiantes que dejaron la escuela sin inscribirse en otra. En los dos últimos años
de la secundaria fueron 11.732 sobre 1.189.142 inscriptos, el
**0,99%**: poco en el total y muy desparejo entre lugares. Decimos que un grupo
tiene **abandono alto** si está en el cuarto de grupos con la tasa más alta del país
(1,74% o más).

## Lo que encontramos, en cinco líneas

Probamos dos modelos que miran solo el perfil (una regresión logística y otro de árboles
combinados) y un tercero que además conoce la provincia. Cuando decimos **el modelo**
hablamos de la regresión logística con el perfil solo. Para medirlos usamos una escala de qué
tan bien ordenan los grupos de más a menos riesgo: **0,5 es lo que lograría un sorteo y 1 es
ordenarlos perfecto**. 0,75 quiere decir que si tomamos un grupo con abandono
alto y uno sin, el modelo pone primero al de abandono alto 3 de cada 4 veces (y
0,80, 4 de cada 5). Además, el modelo **marca** como de
riesgo al cuarto de grupos que pone más arriba: de los 294 grupos con abandono
alto encuentra 138 (47%), y de sus 294 marcas
aciertan 138 (47%).

1. **Con el perfil de los estudiantes se puede ordenar qué grupos tienen más riesgo de
   abandono:** el modelo llega a **0,75**.
2. **La provincia y el perfil aportan información distinta.** Con solo saber en qué
   provincia está el grupo se llega a 0,75, lo mismo que con el perfil.
   Juntos llegan a **0,80**: lo que sabe uno no lo sabe el otro.
3. **Sin conocer la provincia, el perfil dice poco más que la regla simple.** Probado en
   provincias que nunca vio, el modelo llega a 0,68, contra
   0,65 de mirar solo si la escuela es estatal o privada, urbana o rural. La
   diferencia (0,03) es menor que lo que varía el
   resultado de una prueba a otra (0,06). Detalle en la sección 5.
4. **El abandono registrado cambia mucho de una provincia a otra:** de 0,26%
   en San Luis a 3,89% en Misiones, unas
   15 veces más. Creemos que una parte es la forma de completar el
   registro y no abandono real, pero es una hipótesis nuestra: el área de estadística del
   Ministerio de Educación sostiene que el dato es comparable entre provincias (sección 6).
5. **Lo que más acompaña al abandono coincide en buena parte con lo que ya documenta la
   investigación en Argentina:** hogares de nivel socioeconómico bajo, madres que no
   terminaron la primaria, haber repetido, tener más edad de la que corresponde al año,
   haber empezado el jardín recién en sala de 5 y la escuela estatal (sección 7, con las
   fuentes). Es una asociación entre grupos, no una causa.

## Qué significa para quien decide

Que lo que los estudiantes cuentan de sus hogares sirve para ordenar dónde mirar, también
dentro de una misma provincia, aunque no explique por qué pasa. Sumarle la provincia ordena
mejor, pero no sabemos cuánto de lo que aporta la provincia es abandono y cuánto es la forma
de registrarlo: antes de usarla para comparar provincias entre sí, conviene saber cómo
registra el abandono cada una.

## Qué haríamos con esto

- **Mientras no se revise cómo registra cada provincia, usar el perfil para ordenar dentro de
  cada provincia**, y tomar la provincia como un dato a revisar. Es una señal modesta, pero
  es la única que no depende de cómo registra cada provincia: si el abandono alto se define
  dentro de cada provincia (el cuarto de sus grupos con la tasa más alta), el perfil ordena
  en 0,66, contra 0,62 de la regla estatal/privado y
  urbano/rural.
- **Antes de comparar provincias, revisar con ellas cómo registran las salidas sin pase**,
  empezando por las que más se apartan de lo que predice el modelo (Buenos Aires y Santa Fe).
- **Dar una segunda mirada a los grupos que el modelo no marca:** entre ellos quedan
  156 de los 294 con abandono alto, 72 en escuelas
  estatales urbanas y 76 en los dos perfiles rurales (sección 8).

---

Lo que sigue explica cómo llegamos a esto: el problema (1), qué predecimos (2), con qué datos
(3), las reglas simples contra las que nos medimos (4), los modelos (5), dónde se equivocan
(6), qué acompaña al abandono y qué ya se sabía (7), los grupos que el modelo no marca (8),
las limitaciones (9), qué sigue (10), qué es hallazgo nuestro (11) y las fuentes (12). Al
final, cómo reproducirlo. El código vive en el repo del proyecto,
[github.com/IVillanueva770/factor-d10s-m01](https://github.com/IVillanueva770/factor-d10s-m01), y este notebook lo corre fijado al commit
`cb60da0`.

In [ ]:
# Prepara el entorno: baja el proyecto fijado a un commit y corre el pipeline del TP3.
# Tarda alrededor de un minuto. Cada etapa imprime lo que mide.
import os, subprocess, sys
if not os.path.exists("factor-d10s-m01"):
    subprocess.run(["git", "clone", "-q", "https://github.com/IVillanueva770/factor-d10s-m01.git", "factor-d10s-m01"], check=True)
os.chdir("factor-d10s-m01")
subprocess.run(["git", "checkout", "-q", "cb60da02a34065baf1c7463fa7c8f8fb68cd7822"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Proyecto en el commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
      capture_output=True, text=True).stdout.strip())


def etapa(ruta, mostrar=True):
    # Corre una etapa del pipeline y CORTA si falla: el notebook nunca sigue
    # con archivos viejos del repo si una etapa se rompe.
    r = subprocess.run([sys.executable, ruta], capture_output=True, text=True)
    if mostrar:
        print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError("La etapa " + ruta + " falló; el notebook no sigue.")

In [ ]:
# El dataset curado del TP2 se reconstruye desde el maestro del TP1, que está versionado.
etapa("pipeline/03_curacion/08_curar.py", mostrar=False)
print("dataset curado listo")

## 1. El problema y qué representa cada fila

Queremos saber **en qué grupos de escuelas secundarias se concentra el abandono**. La
consigna nos pide primero contestar cinco preguntas, y las contestamos antes de tocar un
modelo:

- **¿Qué representa una fila?** Un grupo de escuelas: un departamento de una provincia,
  separando estatales y privadas, urbanas y rurales. Son 1.174 grupos que juntan a
  540.040 estudiantes que rindieron Aprender. **No es un estudiante**:
  Aprender no se publica estudiante por estudiante.
- **¿Qué queremos predecir?** Si un grupo tiene **abandono alto**: si la proporción de
  estudiantes de los dos últimos años que dejaron la escuela sin inscribirse en otra está en
  el cuarto de grupos con la tasa más alta del país.
- **¿Existe una variable que lo observe?** Sí, y no la inventamos: las **salidas sin pase**
  del Relevamiento Anual, que completa cada escuela. Usamos la definición oficial del área de
  estadística del Ministerio de Educación: salido sin pase es “el que se va de la escuela a la
  que asistía, sin inscripción (o pase) en otra escuela” ([DiNIECE, s/f, p. 7](https://www.argentina.gob.ar/sites/default/files/2018/04/documentos_metodologicos_3.10.zip)). No es el
  abandono interanual oficial, que además cuenta a quienes no se vuelven a inscribir al año
  siguiente; lo retomamos en la sección 10.
- **¿Qué información está disponible antes del resultado?** El perfil de los estudiantes que
  cuenta Aprender, el sector, el ámbito y el tamaño del grupo. **No usamos** nada que se
  conozca después del abandono (los que no pasaron de año, los pases ni los egresados del
  mismo año): salen del mismo cierre de ciclo, y usarlos sería mirar la respuesta.
  Estrictamente, Aprender (octubre de 2024) se tomó después de las salidas que usamos (ciclo
  2023, sección 2): lo que garantizamos es no usar nada del mismo cierre de ciclo, y por eso
  hablamos de ordenar y no de anticipar.
- **¿Qué podemos afirmar?** Cosas sobre grupos, nunca sobre una persona; asociaciones, nunca
  causas.

**Por qué dos fuentes y no una.** Aprender lo responde el estudiante el día de la prueba:
dice cómo le va y cómo vive, pero **no puede ver el abandono**, porque el que ya dejó no va a
rendir. El Relevamiento lo completa la escuela: cuenta quién se fue, pero no sabe nada de los
hogares. Juntas permiten preguntar qué perfiles acompañan a las tasas de abandono que
registra el sistema.

**Un solo año, 2024.** La mentora nos pidió trabajar con una sola época. No hay Aprender de
secundaria 2025 (se toma en 2022 y 2024) ni Relevamiento publicado de 2025, así que todo sale de los archivos 2024 (el del Relevamiento
trae las salidas del ciclo 2023: sección 2). La EPH no sirve para medir el abandono acá: no
se puede unir a los grupos, porque solo llega a provincia, y su dato es de 2025. Queda afuera
del modelo principal y, como pidió la mentora, la probamos aparte (sección 5).

## 2. Qué predecimos, y cómo lo construimos

**La regla, igual para todos los grupos:**

> tasa de abandono = salidas sin pase de los dos últimos años ÷ matrícula inicial de esos
> mismos años. **Abandono alto** = el grupo está en el cuarto de grupos con la tasa más alta
> del país (en estos datos, 1,74% o más).

- **Por qué los dos últimos años.** Son los que evalúa Aprender. La secundaria no dura lo
  mismo en todo el país (en 12 provincias es 7 de primaria y 5 de secundaria, en
  las otras 12 es 6 y 6), pero los dos últimos años caen en el
  mismo lugar en todas.
- **Por qué clases y no el número.** La mentora propuso un enfoque en dos etapas (primero
  separar grupos con y sin abandono alto, después mirar por dentro a los que quedan sin
  marcar), y eso pide clases. Además la tasa tiene muchos ceros: 375 de los
  1.174 grupos (32%) no tienen ninguna salida registrada.
- **Lo que mide en total:** 11.732 salidas sin pase sobre 1.189.142
  inscriptos, el **0,99%**. Poco en el total y muy desparejo entre lugares.

In [ ]:
etapa("pipeline/05_modelado/12_bajar_relevamiento.py")
etapa("pipeline/05_modelado/13_relevamiento.py")

**Dos cosas que medimos y que limitan lo que se puede decir:**

- **Lo que medimos es inestable de un año a otro.** De los grupos con abandono alto en el
  Relevamiento 2024, solo el 53% lo tenía en el anterior (por azar sería
  el 25%; con la secundaria entera sube a 70%). En grupos
  chicos, una sola salida mueve la tasa.
- **El Relevamiento de cada año informa las salidas del ciclo anterior.** Primero lo
  inferimos de los datos (la matrícula de último año del archivo 2024 coincide con la de 2023
  y no con la de 2024) y después lo encontramos documentado: el formulario 2025 pregunta por
  la “Trayectoria de matrícula durante el ciclo lectivo 2024”, con las salidas producidas
  “entre el 30 de abril de 2024 y el último día de clase del período escolar del año 2024”
  ([Relevamiento Anual 2025, cuadernillo nacional, sección 3.12](https://cge.entrerios.gov.ar/wp-content/uploads/cei/2025-COMUN-TP.pdf)). Entonces el Relevamiento 2024 trae las salidas de 2023, y Aprender es
  de octubre de 2024: el perfil se midió después del abandono con el que lo comparamos. Esto
  ordena, no anticipa. Lo usamos igual, porque es el último archivo publicado, y lo dejamos
  declarado (sección 9).

## 3. Los datos y cómo los preparamos

Partimos del dataset curado del TP2 y le sumamos el Relevamiento por la misma clave. El cruce
imprime cuántos estudiantes encuentran su par, y la etapa se corta si se pierden demasiados.

In [ ]:
etapa("pipeline/05_modelado/14_dataset_modelado.py")

**Qué entra al modelo, en bloques que se pueden prender y apagar:**

| Bloque | Qué es |
|---|---|
| Aprender (el perfil) | las proporciones de cada respuesta en el grupo: hogar, educación de la madre y el padre, nivel socioeconómico, repitencia, sobreedad, inasistencias, desempeño en lengua y matemática |
| Contexto | estatal o privado, urbano o rural, tamaño del grupo |
| Edificio (prueba) | proporción de escuelas con electricidad, sala de informática, biblioteca, internet |
| EPH (prueba) | dos indicadores de la provincia: el ingreso per cápita mediano del hogar y la proporción de adultos sin secundaria completa |

Los faltantes (por ejemplo, el clima escolar, que no tiene dato en 96 grupos) se
completan con la mediana **dentro de cada partición de entrenamiento**, nunca mirando los datos
con los que se evalúa. El escalado de las variables se ajusta igual, solo con el entrenamiento.

**Qué dejamos afuera y por qué.** Las columnas de cobertura (miden la calidad del dato, no a los
estudiantes), los indicadores de la EPH (van solo en la prueba aparte), la versión armonizada de
la educación de los padres (repite la original) y todo lo del mismo cierre de ciclo (un test del
proyecto verifica que ninguna de esas columnas entre a ningún experimento). Los tipos ya vienen
revisados del TP2: todas las variables del perfil son proporciones entre 0 y 1, y sector y
ámbito pasan a variables de 0 o 1.

## 4. Las reglas simples contra las que nos medimos

Un modelo solo vale si le gana a una regla que cualquiera podría aplicar sin él. Por eso
comparamos los modelos contra tres reglas:

- **Un sorteo**: a todos el mismo riesgo.
- **Sector y ámbito**: a cada grupo, el riesgo promedio de los grupos de su mismo tipo
  (estatal urbano, estatal rural, privado urbano, privado rural).
- **La provincia**: a cada grupo, el riesgo promedio de su provincia.

## 5. Los modelos, y cómo los comparamos

Probamos una **regresión logística**, que estima la probabilidad de abandono alto sumando el
peso de cada variable y se puede leer, y un modelo de **árboles combinados** (boosting), que
capta relaciones no lineales. Cuando decimos **el modelo** hablamos de la regresión logística
con el perfil solo. Todos, reglas y modelos, se evalúan igual: partimos los datos en cinco y
cada parte se predijo con lo aprendido de las otras cuatro, así que cada grupo lo evaluó un
modelo que no lo había visto (con semilla fija). Usamos cinco partes y no una sola división
porque, con 294 grupos con abandono alto, una sola prueba tendría unos
59 y el resultado dependería mucho de cuáles tocaron; y las
estratificamos para que cada parte tenga la misma proporción de grupos con abandono alto.
Además probamos separando por provincia, para ver qué pasa en provincias que el modelo no vio.

Medimos con el **área bajo la curva ROC**, la escala de qué tan bien ordenan los grupos de más
a menos riesgo (0,5 es un sorteo, 1 es ordenarlos perfecto). Y miramos qué pasa cuando el
modelo **marca** como de riesgo al cuarto de grupos que pone más arriba: de los
294 grupos con abandono alto encuentra 138 (47%,
la cobertura), y de sus 294 marcas aciertan 138
(47%, la precisión). Como marca tantos grupos como hay con abandono alto,
las dos dan lo mismo. No usamos solo el acierto: con un cuarto de grupos con abandono alto,
decir “nadie tiene abandono alto” ya acierta el 75%.

In [ ]:
etapa("pipeline/05_modelado/15_experimentos.py")

In [ ]:
import pandas as pd
r = pd.read_csv("data/processed/resultados_experimentos.csv")
nombres = {"prevalencia": "Regla: un sorteo", "sector_ambito": "Regla: sector y ámbito",
           "provincia": "Regla: la provincia", "logistica": "Regresión logística",
           "boosting": "Árboles combinados"}
t = r[r.experimento == "principal"].assign(modelo=lambda x: x.modelo.map(nombres))
t.pivot(index="modelo", columns="separacion", values="roc_auc").round(2)

**La matriz de confusión del modelo.** Marcamos como de riesgo al cuarto de grupos que el modelo
pone más arriba, con todas las predicciones fuera de muestra juntas y un solo corte: es la que
usamos en el texto. (La tabla de experimentos calcula las métricas dentro de cada partición, con
un corte por parte, y por eso puede diferir en unos pocos grupos.)

In [ ]:
import sys
sys.path.insert(0, "pipeline")
from modelado import EXPERIMENTOS, cuantil_de, target_de
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
principal = next(e for e in EXPERIMENTOS if e["id"] == "principal")
p = pd.read_csv("data/processed/predicciones_principal.csv")
y = p[target_de(principal)].astype(int)
marca = (p["p_logistica"] >= p["p_logistica"].quantile(cuantil_de(principal))).astype(int)
display(pd.DataFrame(confusion_matrix(y, marca),
                     index=["real: sin abandono alto", "real: abandono alto"],
                     columns=["no marcado", "marcado"]))
pd.Series({"precisión": precision_score(y, marca),
           "cobertura (recall)": recall_score(y, marca),
           "F1": f1_score(y, marca),
           "PR-AUC (azar = 0,25)": average_precision_score(y, p["p_logistica"]),
           "ROC-AUC": roc_auc_score(y, p["p_logistica"])}).round(2)

In [ ]:
from IPython.display import Image
etapa("pipeline/05_modelado/18_figuras.py", mostrar=False)
Image("figuras/tp3_a_modelo_vs_reglas.png")

**Lo que dice la tabla:**

- Con todas las provincias a la vista, el modelo llega a **0,75**, la regla de
  la provincia a **0,75** y la de sector y ámbito a **0,66**.
  El perfil y la provincia empatan. El de árboles combinados, más complejo, no mejora
  (0,73): señal de que el patrón es simple.
- Si probamos el modelo en provincias que no usó para aprender, la regla de la provincia queda
  sin nada que decir y el modelo llega a **0,68**, apenas por encima de la
  regla de sector y ámbito (**0,65**). La diferencia
  (0,03) es menor que lo que varía el resultado de una
  prueba a otra (0,06). **Sin la provincia, el perfil dice poco más
  que la regla simple.**

**La provincia y el perfil juntos.** Empatar no es saber lo mismo. Para verlo armamos un único
modelo que conoce las dos cosas (una columna por provincia, más el perfil): llega a
**0,80** (con árboles combinados, 0,78). Si la
provincia solo resumiera el perfil de sus estudiantes, sumarla no agregaría nada; agrega, así
que **cada uno aporta información que el otro no tiene**. Lo probamos así y no con un modelo
por provincia porque cada provincia tiene unos 49 grupos y unos
12 con abandono alto: no alcanza para entrenar ni para evaluar un modelo
propio.

**Las combinaciones que probamos además** (todas en la tabla de abajo):

- **Mover el corte de abandono alto.** Elegir el cuarto de grupos con más abandono como
  “alto” fue decisión nuestra, así que lo movimos al tercio y al quinto: el perfil solo y la
  provincia sola siguen parejos (en los dos casos, 0,74 contra
  0,75).
- **Sumar el edificio** lleva la regresión a 0,76: casi nada.
- **Sumar la EPH.** Como pidió la mentora, probamos sumar sus dos indicadores por provincia.
  Con la regresión logística no cambia (0,75), pero con árboles combinados sube
  de 0,73 a 0,77: dos datos por provincia alcanzan para que
  el modelo reconozca la provincia, que es la información que el perfil no tiene. Lo que la
  provincia agrega por encima del perfil, estos datos no lo nombran: puede ser algo propio de
  cada provincia o su forma de registrar (sección 6).
- **Con la secundaria entera** como target, la regla de la provincia sube a
  0,84 y el modelo queda en 0,79: ahí la provincia
  pesa todavía más.
- **Con el corte dentro de cada provincia** (el cuarto de sus grupos con la tasa más alta), el
  perfil ordena en 0,66 contra 0,62 de sector y ámbito. Es
  la señal que no depende de cómo registra cada provincia, y por eso es la que recomendamos
  usar mientras eso no se revise.

In [ ]:
t = r[r.separacion == "estratificada"].assign(modelo=lambda x: x.modelo.map(nombres))
t.pivot(index="experimento", columns="modelo", values="roc_auc").round(2)

## 6. Dónde se equivoca: abandono o forma de registrar

Miramos los errores del modelo por tamaño del grupo, por sector y ámbito y por provincia.

In [ ]:
etapa("pipeline/05_modelado/16_errores.py")

- **Los errores no se reparten parejo entre provincias, y eso es lo más importante.** En
  Buenos Aires hay 22 grupos con abandono alto y el modelo detecta 1,
  pero marca otros 17 que el registro da como bajos: **por el perfil de sus
  estudiantes, esos grupos “deberían” tener más abandono del que figura.** En Santa Fe pasa
  al revés: 34 grupos con abandono alto y el modelo detecta 7. O cada
  provincia registra distinto, o hay algo propio de cada una que Aprender no ve; con estos
  datos no lo podemos distinguir.
- **Que el registro cambie entre provincias es una hipótesis nuestra.** El área de estadística
  del Ministerio de Educación, que administra el Relevamiento, sostiene lo contrario: que su
  sistema “garantiza la homogeneidad y comparabilidad de los datos en todo el ámbito
  nacional” ([DIEE, s/f](https://www.argentina.gob.ar/sites/default/files/documento_metodologico.pdf)). No encontramos ningún documento oficial que advierta diferencias.
  Nos apoyamos en la distancia entre provincias (de 0,26% en
  San Luis a 3,89% en Misiones) y en la caída de Buenos
  Aires, que con la secundaria entera pasa de 2,97% en 2011 a
  0,30% en 2024. Por eso proponemos revisarlo con las
  provincias antes de compararlas.
- **El privado urbano casi no tiene abandono alto**, y el modelo lo sabe: ahí casi no se
  equivoca.
- **El modelo detecta peor a los grupos más grandes.** En el quinto de grupos más grandes
  encuentra el 34% de los que tienen abandono alto, contra el
  47% en el quinto más chico: ahí las tasas son más bajas y parejas, y
  lo poco que pasa queda diluido.

In [ ]:
Image("figuras/tp3_b_tasa_por_provincia.png")

## 7. Qué acompaña al abandono, y qué ya se sabía

La lista de qué acompaña al abandono no sale de los pesos de la regresión. Para ver hacia
dónde empuja cada respuesta, miramos su correlación con la tasa de abandono; para ver cuánto
aporta cada tema, sacamos un bloque entero de variables, volvemos a evaluar y miramos cuánto
empeora el modelo (la caída va en centésimas de la escala de 0,5 a 1).

In [ ]:
a = pd.read_csv("data/processed/aporte_bloques.csv")
a.assign(caida_en_centesimas=100 * a.caida)[
    ["bloque", "auc_sin_el_bloque", "caida_en_centesimas"]].round(2)

In [ ]:
s = pd.read_csv("data/processed/sentido_variables.csv")
pd.concat([s.head(6), s.tail(6)]).round(2)

- **El sentido.** En nuestros datos hay más abandono donde hay más hogares de nivel
  socioeconómico bajo, más madres que no terminaron la primaria, más estudiantes que
  repitieron o que tienen más edad de la que corresponde al año, más que empezaron el jardín
  recién en sala de 5, y en las escuelas estatales. Menos abandono donde los estudiantes tienen
  la edad que corresponde, nunca repitieron y fueron al jardín desde sala de 3.
- **Ningún tema por separado es imprescindible para el modelo.** Sacar cualquier bloque mueve
  el resultado 1,04 centésimas como mucho: al sacar uno, los otros lo
  cubren, porque todos cuentan la misma desigualdad.
- **Qué de esto ya estaba documentado.** No lo dimos por sentado: buscamos qué dice la
  investigación previa.
  - *Nivel socioeconómico y escuela estatal: documentado.* Un estudio que siguió veinte años
    de la EPH encuentra más riesgo de abandono en los hogares más pobres (“los estudiantes de
    los quintiles más pobres tienen más probabilidad de entrar en ese grupo”) y en las
    escuelas de gestión estatal ([Adrogué y Orlicki, 2025](https://ri.conicet.gov.ar/bitstream/handle/11336/267725/CONICET_Digital_Nro.edd2aab9-34e4-449f-a0c2-93e0f205899b_B.pdf?sequence=2)).
  - *Educación de la madre: documentado.* La encuesta de UNICEF Argentina sobre
    adolescencias lo muestra como “clima educativo del hogar” ([UNICEF Argentina, 2022](https://www.unicef.org/argentina/media/18746/file/Serie%20MICS:%20Adolescencias%20y%20Educaci%C3%B3n.pdf), p. 22). Ojo: en
    Aprender 2024 el índice de nivel socioeconómico ya incluye la educación de la madre (se
    construye “considerando diversas variables como el nivel educativo de la madre, el
    hacinamiento”, [Aprender 2024](https://www.argentina.gob.ar/sites/default/files/informe_nacional_aprender_2024.pdf), p. 30), así que en nuestros datos no son dos señales
    independientes.
  - *Repitencia y sobreedad: documentado con dato oficial.* El área de estadística del
    Ministerio de Educación mostró en 2024 un abandono del 27% entre quienes tienen sobreedad
    contra 4,5% entre quienes no, con otra forma de medir el abandono, que sigue a cada
    estudiante a lo largo de los años, en 11 provincias ([Caminos interrumpidos, 2024](https://www.argentina.gob.ar/sites/default/files/historias_con_datos_sobredad_final.pdf), p. 5). Y el informe
    de Aprender dice que repetir “en muchos casos, se asocia con un mayor riesgo de abandono
    escolar” ([Aprender 2024](https://www.argentina.gob.ar/sites/default/files/informe_nacional_aprender_2024.pdf), p. 46).
  - *Jardín desde sala de 3: documentado en parte.* El informe nacional de Aprender 2024 lo
    asocia a mejores aprendizajes ([Aprender 2024](https://www.argentina.gob.ar/sites/default/files/informe_nacional_aprender_2024.pdf), p. 45), pero no encontramos una fuente
    argentina que lo mida contra el abandono.
- **Un resultado raro que ya conocíamos**: los grupos donde la mayoría falta poco tienen
  *más* abandono, no menos. Es el sesgo de selección del TP2: Aprender solo ve a quien fue a
  rendir, y el que más faltaba probablemente ya se fue.
- **Que algo acompañe al abandono no lo vuelve causa.** Sirve para saber dónde mirar; qué
  cambiar es otra pregunta.

## 8. Los grupos que el modelo no marca

La mentora propuso una segunda mirada: dentro de los grupos que el modelo no marca como
abandono alto, ¿hay perfiles distintos? Los agrupamos por seis rasgos (nivel socioeconómico,
educación de la madre, repitencia, desempeño en matemática, sector y ámbito), sin usar el
abandono para armarlos, con un agrupamiento k-means, y aparecen 4 perfiles.

In [ ]:
etapa("pipeline/05_modelado/17_perfiles.py")
Image("figuras/tp3_c_perfiles.png")

Lo que el modelo no marca no es todo igual. El **privado urbano** casi no tiene abandono. Los
otros tres (**estatal urbano**, **rural estatal** y **rural más vulnerable**) tienen tasas
parecidas entre sí. Entre todos los no marcados quedan 156 de los
294 grupos con abandono alto: 72
(46%) en el estatal urbano y 76 entre los dos
rurales.

**Para quien decide:** el estatal urbano junta más casos por su tamaño (reúne el
58% de los estudiantes de estos grupos); el rural más vulnerable tiene
la tasa más alta.

## 9. Limitaciones

- **Grupos, no personas.** Nada de esto permite decir algo de un estudiante concreto.
- **El abandono registrado no es el abandono real.** Un cero no se distingue de “no se
  registró”, y hay muchos: 375 de los 1.174 grupos
  (32%) no tienen ninguna salida registrada. Que la diferencia entre
  provincias sea por cómo registra cada una es una hipótesis nuestra (sección 6).
- **Lo que el registro no cuenta y lo que cuenta de más.** Quien se va entre diciembre y
  marzo, después del último día de clase, no figura como salida sin pase, según el formulario
  2025 del Relevamiento ([Relevamiento Anual 2025, cuadernillo nacional, sección 3.12](https://cge.entrerios.gov.ar/wp-content/uploads/cei/2025-COMUN-TP.pdf)); y la categoría incluye a quienes siguen en otra
  escuela sin haber hecho el trámite del pase.
- **Un año de diferencia.** El Relevamiento de cada año informa las salidas del ciclo
  anterior: el de 2024 trae las de 2023, y Aprender es de octubre de 2024. Es decir, el perfil
  se midió después del abandono con el que lo comparamos: esto ordena, no anticipa. Anticipar
  es el próximo paso.
- **Lo que medimos es inestable.** De los grupos con abandono alto en el Relevamiento 2024,
  solo el 53% lo tenía en el anterior (por azar sería el
  25%): en grupos chicos, una sola salida mueve la tasa.
- **El modelo detecta peor a los grupos más grandes.** En el quinto de grupos más grandes
  encuentra el 34% de los que tienen abandono alto, contra el
  47% en el quinto más chico: ahí las tasas son más bajas y parejas, y
  lo poco que pasa queda diluido.
- **Aprender solo ve a quien fue a rendir**, y eso subestima justo a los grupos con más
  abandono.
- **El corte del cuarto superior es una decisión nuestra.** Lo probamos con el tercio y el
  quinto y aguanta: el perfil solo y la provincia sola siguen parejos (sección 5).
- **El edificio** del Relevamiento cuenta todas las escuelas del grupo, no solo las
  secundarias.

## 10. Qué sigue

- **Sumar el Relevamiento 2025**, que trae el abandono del ciclo 2024, el mismo año de
  Aprender.
- **Incorporar el abandono interanual oficial** (quiénes no se reinscriben al año siguiente),
  que capta lo que las salidas sin pase no ven.
- **Revisar con las provincias cómo registran las salidas sin pase**, empezando por Buenos
  Aires y Santa Fe.
- **Predecir el año siguiente con los datos del anterior**, que es lo que haría un sistema de
  alerta temprana.

## 11. Qué es hallazgo nuestro y qué ya se sabía

- **Ya se sabía**: que el nivel socioeconómico bajo, la escuela estatal, el clima educativo del
  hogar, la repitencia y la sobreedad acompañan al abandono; qué es una salida sin pase, y que
  el Relevamiento la informa para el ciclo anterior.
- **Se sabía en parte**: lo del jardín (documentado para los aprendizajes, no contra el
  abandono).
- **Es hallazgo nuestro** (no lo encontramos dicho en las fuentes):
  - que, con datos agregados por grupo, la provincia sola ordene el riesgo igual que el perfil
    de los estudiantes (0,75 y 0,75), y que juntos lleguen a
    0,80;
  - que sin conocer la provincia el perfil diga poco más que la regla estatal/privado y
    urbano/rural (0,68 contra 0,65, una diferencia menor
    que la variación entre pruebas);
  - que dos indicadores de la EPH por provincia no le agreguen nada a la regresión logística
    pero le alcancen al modelo de árboles combinados para reconocer la provincia;
  - la asociación invertida de las inasistencias, que leemos como sesgo de selección de
    Aprender (TP2);
  - la hipótesis de que las provincias registran distinto las salidas sin pase, que el área de
    estadística del Ministerio de Educación no comparte.

## 12. Fuentes

- Adrogué, C. y Orlicki, E. (2025). Riesgo de abandono escolar en la secundaria. Factores comunes y diferentes para varones y mujeres. Estudios Económicos 42(85), 58-79. Universidad Nacional del Sur. [Enlace](https://ri.conicet.gov.ar/bitstream/handle/11336/267725/CONICET_Digital_Nro.edd2aab9-34e4-449f-a0c2-93e0f205899b_B.pdf?sequence=2)
- Dirección Nacional de Información y Evaluación de la Calidad Educativa (DiNIECE), Ministerio de Educación, Ciencia y Tecnología (s/f). Definiciones básicas para la producción de estadísticas educativas. [Enlace](https://www.argentina.gob.ar/sites/default/files/2018/04/documentos_metodologicos_3.10.zip)
- Dirección de Información y Estadística Educativa (DIEE) (s/f). Bases Usuarias Relevamiento Anual. Documento metodológico. [Enlace](https://www.argentina.gob.ar/sites/default/files/documento_metodologico.pdf)
- Relevamiento Anual 2025. Educación Común. Matrícula y cargos (cuadernillo nacional, 2025). [Enlace](https://cge.entrerios.gov.ar/wp-content/uploads/cei/2025-COMUN-TP.pdf)
- Subsecretaría de Información y Evaluación Educativa, Ministerio de Capital Humano (2025). Informe de resultados 2024. Educación Secundaria (Aprender 2024). [Enlace](https://www.argentina.gob.ar/sites/default/files/informe_nacional_aprender_2024.pdf)
- Subsecretaría de Información y Evaluación Educativa, Ministerio de Capital Humano (2024). Caminos interrumpidos: abandono escolar y sobreedad. Historias con datos. [Enlace](https://www.argentina.gob.ar/sites/default/files/historias_con_datos_sobredad_final.pdf)
- UNICEF Argentina (2022). Encuesta Nacional de Niñas, Niños y Adolescentes. MICS 2019-2020. Informe temático: Adolescencias y educación. [Enlace](https://www.unicef.org/argentina/media/18746/file/Serie%20MICS:%20Adolescencias%20y%20Educaci%C3%B3n.pdf)

---

## Cómo reproducirlo

Cada número de este notebook lo produce el pipeline del proyecto (`pipeline/05_modelado/`),
con las decisiones en un solo archivo (`pipeline/modelado.py`) y verificaciones automáticas en
`tests/test_invariantes.py`. Para probar otra combinación alcanza con editar la lista de
experimentos y volver a correr desde la etapa 15. Las decisiones y sus motivos están en
`entregas/03-machine-learning/DECISIONES-TP3.md`, y cada afirmación que no sale de nuestros
datos tiene su cita textual y su enlace en `entregas/03-machine-learning/FUENTES-TP3.md`.